# SprintBoard GRPO Training (Google Colab)

This notebook trains the SprintBoard agent using the existing `train_grpo.py` / `train_grpo_unsloth.py` scripts in this repo.

Run cells top-to-bottom. A GPU runtime is strongly recommended.

## 1) Setup runtime and repo

If this notebook is outside the repository, keep `CLONE_REPO = True`.
If this notebook is already inside the repo in Colab, set it to `False`.

In [ ]:
import os
from pathlib import Path

CLONE_REPO = True
REPO_URL = "https://github.com/<your-org-or-user>/sprint_planning_env.git"  # TODO: update
WORKDIR = Path('/content/sprint_planning_env')

if CLONE_REPO:
    if WORKDIR.exists():
        !rm -rf /content/sprint_planning_env
    !git clone {REPO_URL} {WORKDIR}
else:
    WORKDIR = Path.cwd()

%cd {WORKDIR}
print('Working directory:', WORKDIR)


In [ ]:
# Optional: verify GPU
!nvidia-smi || true


## 2) Install dependencies

In [ ]:
!pip -q install -U pip
!pip -q install -r requirements-train.txt

# Force upgrade Unsloth + unsloth_zoo to a fixed compatible version (issue #4885)
!pip -q install --upgrade --force-reinstall --no-deps "unsloth>=2026.4.5" "unsloth_zoo>=2026.4.4"

# Runtime deps required by SprintBoard environment used in reward functions
!pip -q install -r requirements.txt

# Make local package importable in Colab
!pip -q install -e .


In [ ]:
# Clear stale Unsloth compile cache (can break with TRL/Unsloth updates)
!rm -rf /content/sprint_planning_env/unsloth_compiled_cache

# Verify you are running the latest patched training script
!python - <<'PYCHK'
from pathlib import Path
p = Path('train_grpo_unsloth.py')
t = p.read_text(encoding='utf-8')
checks = [
    ('import unsloth', 'import unsloth' in t),
    ('all-linear target', 'target_modules=["all-linear"]' in t),
    ('lora-dropout default 0.0', 'lora-dropout", type=float, default=0.0' in t),
]
print('train_grpo_unsloth.py checks:')
for name, ok in checks:
    print(f'  - {name}:', 'OK' if ok else 'MISSING')
if not all(ok for _, ok in checks):
    raise SystemExit('Script is stale. Re-clone repo / restart runtime and rerun setup cells.')
PYCHK


## 3) (Optional) Hugging Face login
Only needed if you use private models or want to push checkpoints.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()


## 4) Configure training
Tune these values based on your GPU memory/time budget.

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
OUTPUT_DIR = 'runs/colab-grpo-unsloth'
EPOCHS = 1
BATCH_SIZE = 1
GRAD_ACCUM = 4
NUM_GENERATIONS = 4
LEARNING_RATE = 5e-6
MAX_SAMPLES = 15
MAX_PROMPT_LENGTH = 512
MAX_COMPLETION_LENGTH = 64
LOGGING_STEPS = 1


## 5) Train (Unsloth only)


In [ ]:
!python train_grpo_unsloth.py \
  --model-id "{MODEL_ID}" \
  --output-dir "{OUTPUT_DIR}" \
  --epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --grad-accum {GRAD_ACCUM} \
  --num-generations {NUM_GENERATIONS} \
  --learning-rate {LEARNING_RATE} \
  --max-samples {MAX_SAMPLES} \
  --max-prompt-length {MAX_PROMPT_LENGTH} \
  --max-completion-length {MAX_COMPLETION_LENGTH} \
  --logging-steps {LOGGING_STEPS} \
  --lora-dropout 0.0


## 7) Evaluate checkpoint (optional)

In [ ]:
CHECKPOINT_DIR = f'{OUTPUT_DIR}/checkpoint-final'
!python evaluate_training.py --output-dir runs/colab-eval --checkpoint-dir "{CHECKPOINT_DIR}"


## 8) Inspect artifacts

In [ ]:
from pathlib import Path

summary_path = Path(OUTPUT_DIR) / 'artifacts' / 'summary.json'
if summary_path.exists():
    print(summary_path.read_text())
else:
    print('No summary.json found yet.')

!ls -lah {OUTPUT_DIR}/artifacts || true
!ls -lah runs/colab-eval || true


## 9) (Optional) Save outputs to Google Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/sprintboard-runs
# !cp -r {OUTPUT_DIR} /content/drive/MyDrive/sprintboard-runs/
# !cp -r runs/colab-eval /content/drive/MyDrive/sprintboard-runs/
